# Name That LLM — CPU submission

由 scripts/build_submission_notebook.py 生成，请修改权威源码后重新生成。仅含全量拟合、运行时预测及严格写后校验。CPU，Internet Off；无需本地项目或模型。本地验收不等于 Kaggle 运行或提交成功。

官方依据：[比赛概览](https://www.kaggle.com/competitions/name-that-llm/overview/description)、[数据](https://www.kaggle.com/competitions/name-that-llm/data)。Rules、账号资格、团队和配额请在正式提交前人工确认。

## 1 依赖和环境

使用环境已安装的依赖，不下载、不安装。打印实际版本；配置不兼容时明确失败。

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import platform
import time
import warnings
from copy import deepcopy
import numpy as np
import pandas as pd
import pyarrow
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

runtime_versions = {name: importlib.metadata.version(name)
                    for name in ('numpy', 'pandas', 'scikit-learn', 'scipy', 'pyarrow')}
print('ENVIRONMENT', json.dumps({'python': platform.python_version(), **runtime_versions}))


## 2 数据与输出路径

优先显式 NAME_THAT_LLM_DATA_DIR，再查官方挂载；发现多个候选时要求明确指定。Kaggle 输出目录为 /kaggle/working。本地请在新目录执行或设置 NAME_THAT_LLM_OUTPUT_DIR。

In [ ]:
KAGGLE_DATA = Path('/kaggle/input/competitions/name-that-llm')


REQUIRED_DATA_FILES = ('train.csv', 'test.csv')


def resolve_submission_data(explicit_dir=None):
    def missing(directory):
        return [name for name in REQUIRED_DATA_FILES if not (directory / name).is_file()]
    if explicit_dir is not None:
        if not str(explicit_dir).strip():
            raise ValueError('NAME_THAT_LLM_DATA_DIR cannot be empty')
        directory = Path(explicit_dir).expanduser().resolve()
        absent = missing(directory)
        if absent:
            raise FileNotFoundError(f'Explicit data directory {directory}: missing {absent}; no fallback used')
        return directory
    if not missing(KAGGLE_DATA):
        return KAGGLE_DATA.resolve()
    # A mount can differ between notebook environments. Do not guess among datasets.
    input_root = Path('/kaggle/input')
    candidates = sorted({p.parent.resolve() for p in input_root.rglob('train.csv')
                         if not missing(p.parent)}) if input_root.is_dir() else []
    if len(candidates) == 1:
        print('DISCOVERED_INPUT_PAIR; training SHA-256 will be checked:', candidates[0])
        return candidates[0]
    raise FileNotFoundError(
        f'Competition inputs unavailable or ambiguous. Official candidate {KAGGLE_DATA}: '
        f'missing {missing(KAGGLE_DATA)}; complete candidate directories={list(map(str, candidates))}. '
        'Attach the official competition data and inspect Input; set NAME_THAT_LLM_DATA_DIR explicitly if needed.')
DATA_DIR = resolve_submission_data(os.environ.get('NAME_THAT_LLM_DATA_DIR'))
output_override = os.environ.get('NAME_THAT_LLM_OUTPUT_DIR')
if output_override is not None and not output_override.strip():
    raise ValueError('NAME_THAT_LLM_OUTPUT_DIR cannot be empty')
OUTPUT_DIR = (Path(output_override).expanduser() if output_override is not None else
              Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()).resolve()
print('DATA_DIR', DATA_DIR)
print('OUTPUT_DIR', OUTPUT_DIR)


## 3 输入契约和官方标签

标签和输入函数从公共模块提取；保留原始文本、字符串 ID 和顺序。训练 SHA 是已验证数据版本检查；测试不限制行数/ID/指纹，也不读取样例占位值。

In [ ]:
OFFICIAL_MODEL_LABELS = ('gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3')


class_order = tuple(sorted(OFFICIAL_MODEL_LABELS))
def _locations(frame, mask):
    positions = np.flatnonzero(np.asarray(mask))[:5]
    return [{'record': int(i) + 1,
             'id': repr(frame.iloc[i]['id']) if 'id' in frame else '<missing column>'}
            for i in positions]


def _validate_fields(frame, required, source):
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f'{source}: missing required fields {missing}')
    for field in required:
        values = frame[field]
        null = values.isna()
        if null.any():
            raise ValueError(f'{source}: field {field!r} contains null; records {_locations(frame, null)}')
        non_string = ~values.map(lambda value: isinstance(value, str)).astype(bool)
        if non_string.any():
            raise ValueError(f'{source}: field {field!r} must contain strings; records {_locations(frame, non_string)}')
        blank = values.str.strip().eq('')
        if blank.any():
            raise ValueError(f'{source}: field {field!r} is empty/whitespace; records {_locations(frame, blank)}')
    duplicate = frame['id'].duplicated(keep=False)
    if duplicate.any():
        raise ValueError(f'{source}: duplicate id; records {_locations(frame, duplicate)}')


def validate_train(frame, source='<train>', require_all_labels=True):
    _validate_fields(frame, ('id', 'prompt_id', 'text', 'model'), source)
    actual = set(frame['model'])
    unknown = sorted(actual - set(class_order))
    missing = sorted(set(class_order) - actual) if require_all_labels else []
    if unknown or missing:
        invalid = ~frame['model'].isin(class_order)
        raise ValueError(f'{source}: field model label mismatch; unknown={unknown}; '
                         f'missing={missing}; invalid records={_locations(frame, invalid)}')
    return frame


def validate_test(frame, source='<test>'):
    _validate_fields(frame, ('id', 'text'), source)
    return frame


def read_input_csv(path, kind, require_all_labels=True):
    path = Path(path).expanduser().resolve()
    if kind not in ('train', 'test'):
        raise ValueError(f'{path}: unknown input kind {kind!r}')
    try:
        # Empty fields remain empty; NA/NULL/nan are literal strings, not null markers.
        frame = pd.read_csv(path, dtype='string', keep_default_na=False,
                            na_filter=False, skip_blank_lines=False, encoding='utf-8-sig')
    except (OSError, ValueError, pd.errors.ParserError) as exc:
        raise ValueError(f'{path}: could not read {kind} CSV: {exc}') from exc
    if kind == 'train':
        return validate_train(frame, str(path), require_all_labels)
    return validate_test(frame, str(path))


def file_fingerprint(path):
    path = Path(path).resolve()
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return {'path': str(path), 'size_bytes': path.stat().st_size, 'sha256': digest.hexdigest()}

EXPECTED_TRAIN_SHA256 = '40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707'
train_path, test_path = DATA_DIR / 'train.csv', DATA_DIR / 'test.csv'
train_fingerprint = file_fingerprint(train_path)
if train_fingerprint['sha256'] != EXPECTED_TRAIN_SHA256:
    raise ValueError('Official training version differs from S04; inspect data attachment/update before changing the baseline')
train = read_input_csv(train_path, 'train')
test = read_input_csv(test_path, 'test')
if len(train) != 2699 or train['prompt_id'].nunique() != 150:
    raise ValueError('S04 official training version expects 2699 records and 150 prompts')
test_fingerprint = file_fingerprint(test_path)
print('INPUT_CHECKED', len(train), train['prompt_id'].nunique(), len(class_order), 'test rows', len(test))


## 4 创建已验证基线

唯一模型工厂和配置来自公共模块；EXPECTED_MODEL_PARAMETERS 为 S04 生成时的完整配置断言，不用于另建模型。若依赖导致参数差异，停止并报告，不静默替换。

In [ ]:
_FEATURE_DEFAULTS = {
    'word': {'analyzer': 'word', 'ngram_range': [1, 2], 'min_df': 2,
             'max_features': 40000, 'sublinear_tf': True, 'dtype': 'float32',
             'lowercase': True, 'norm': 'l2', 'use_idf': True, 'smooth_idf': True, 'max_df': 1.0},
    'char': {'analyzer': 'char', 'ngram_range': [2, 5], 'min_df': 2,
             'max_features': 80000, 'sublinear_tf': True, 'dtype': 'float32',
             'lowercase': True, 'norm': 'l2', 'use_idf': True, 'smooth_idf': True, 'max_df': 1.0},
}


def feature_defaults():
    return deepcopy(_FEATURE_DEFAULTS)


def make_features(config):
    transforms = []
    for name in config['branches']:
        settings = deepcopy(config['features'][name])
        if settings.pop('dtype') != 'float32':
            raise ValueError('Only the verified float32 TF-IDF dtype is supported')
        settings['ngram_range'] = tuple(settings['ngram_range'])
        transforms.append((name, TfidfVectorizer(dtype=np.float32, **settings)))
    return FeatureUnion(transforms)
RANDOM_SEED = 20261118


def baseline_config():
    return {'branches': ['word', 'char'], 'features': feature_defaults(),
            'classifier': {'C': 1.0, 'class_weight': 'balanced', 'max_iter': 3000,
                'tol': 0.0001, 'penalty': 'l2', 'loss': 'squared_hinge', 'multi_class': 'ovr',
                'dual': 'auto', 'fit_intercept': True, 'intercept_scaling': 1,
                'verbose': 0, 'random_state': RANDOM_SEED}}


def resolve_config(overrides=None):
    result = baseline_config()
    def merge(target, supplied, prefix=''):
        if not isinstance(supplied, dict):
            raise ValueError(f'{prefix or "config"} must be a dictionary')
        for key, value in supplied.items():
            if key not in target:
                raise ValueError(f'Unknown experiment setting: {prefix}{key}')
            if isinstance(target[key], dict):
                merge(target[key], value, prefix + key + '.')
            else:
                target[key] = deepcopy(value)
    if overrides is not None:
        merge(result, overrides)
    branches = result['branches']
    if (not isinstance(branches, list) or not branches or len(branches) != len(set(branches))
            or not set(branches).issubset({'word', 'char'})):
        raise ValueError('branches must be a nonempty unique list containing word and/or char')
    return result


def make_model(config=None):
    settings = resolve_config(config)
    return Pipeline([('features', make_features(settings)),
                     ('classifier', LinearSVC(**settings['classifier']))])
def normalise(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, (list, tuple)):
        return [normalise(item) for item in value]
    if isinstance(value, dict):
        return {str(key): normalise(item) for key, item in value.items()}
    if isinstance(value, type):
        return value.__module__ + '.' + value.__name__
    return repr(value)


def model_config(model):
    return normalise(model.get_params(deep=True))

EXPECTED_MODEL_PARAMETERS = {'memory': None, 'steps': [['features', "FeatureUnion(transformer_list=[('word',\n                                TfidfVectorizer(dtype=<class 'numpy.float32'>,\n                                                max_features=40000, min_df=2,\n                                                ngram_range=(1, 2),\n                                                sublinear_tf=True)),\n                               ('char',\n                                TfidfVectorizer(analyzer='char',\n                                                dtype=<class 'numpy.float32'>,\n                                                max_features=80000, min_df=2,\n                                                ngram_range=(2, 5),\n                                                sublinear_tf=True))])"], ['classifier', "LinearSVC(class_weight='balanced', max_iter=3000, random_state=20261118)"]], 'transform_input': None, 'verbose': False, 'features': "FeatureUnion(transformer_list=[('word',\n                                TfidfVectorizer(dtype=<class 'numpy.float32'>,\n                                                max_features=40000, min_df=2,\n                                                ngram_range=(1, 2),\n                                                sublinear_tf=True)),\n                               ('char',\n                                TfidfVectorizer(analyzer='char',\n                                                dtype=<class 'numpy.float32'>,\n                                                max_features=80000, min_df=2,\n                                                ngram_range=(2, 5),\n                                                sublinear_tf=True))])", 'classifier': "LinearSVC(class_weight='balanced', max_iter=3000, random_state=20261118)", 'features__n_jobs': None, 'features__transformer_list': [['word', "TfidfVectorizer(dtype=<class 'numpy.float32'>, max_features=40000, min_df=2,\n                ngram_range=(1, 2), sublinear_tf=True)"], ['char', "TfidfVectorizer(analyzer='char', dtype=<class 'numpy.float32'>,\n                max_features=80000, min_df=2, ngram_range=(2, 5),\n                sublinear_tf=True)"]], 'features__transformer_weights': None, 'features__verbose': False, 'features__verbose_feature_names_out': True, 'features__word': "TfidfVectorizer(dtype=<class 'numpy.float32'>, max_features=40000, min_df=2,\n                ngram_range=(1, 2), sublinear_tf=True)", 'features__char': "TfidfVectorizer(analyzer='char', dtype=<class 'numpy.float32'>,\n                max_features=80000, min_df=2, ngram_range=(2, 5),\n                sublinear_tf=True)", 'features__word__analyzer': 'word', 'features__word__binary': False, 'features__word__decode_error': 'strict', 'features__word__dtype': 'numpy.float32', 'features__word__encoding': 'utf-8', 'features__word__input': 'content', 'features__word__lowercase': True, 'features__word__max_df': 1.0, 'features__word__max_features': 40000, 'features__word__min_df': 2, 'features__word__ngram_range': [1, 2], 'features__word__norm': 'l2', 'features__word__preprocessor': None, 'features__word__smooth_idf': True, 'features__word__stop_words': None, 'features__word__strip_accents': None, 'features__word__sublinear_tf': True, 'features__word__token_pattern': '(?u)\\b\\w\\w+\\b', 'features__word__tokenizer': None, 'features__word__use_idf': True, 'features__word__vocabulary': None, 'features__char__analyzer': 'char', 'features__char__binary': False, 'features__char__decode_error': 'strict', 'features__char__dtype': 'numpy.float32', 'features__char__encoding': 'utf-8', 'features__char__input': 'content', 'features__char__lowercase': True, 'features__char__max_df': 1.0, 'features__char__max_features': 80000, 'features__char__min_df': 2, 'features__char__ngram_range': [2, 5], 'features__char__norm': 'l2', 'features__char__preprocessor': None, 'features__char__smooth_idf': True, 'features__char__stop_words': None, 'features__char__strip_accents': None, 'features__char__sublinear_tf': True, 'features__char__token_pattern': '(?u)\\b\\w\\w+\\b', 'features__char__tokenizer': None, 'features__char__use_idf': True, 'features__char__vocabulary': None, 'classifier__C': 1.0, 'classifier__class_weight': 'balanced', 'classifier__dual': 'auto', 'classifier__fit_intercept': True, 'classifier__intercept_scaling': 1, 'classifier__loss': 'squared_hinge', 'classifier__max_iter': 3000, 'classifier__multi_class': 'ovr', 'classifier__penalty': 'l2', 'classifier__random_state': 20261118, 'classifier__tol': 0.0001, 'classifier__verbose': 0}
try:
    probe = make_model()
    actual_parameters = model_config(probe)
except (TypeError, ValueError) as exc:
    raise RuntimeError(f'Baseline construction incompatible with environment {runtime_versions}: {exc}') from exc
if actual_parameters != EXPECTED_MODEL_PARAMETERS:
    differences = sorted(k for k in set(actual_parameters) | set(EXPECTED_MODEL_PARAMETERS)
                         if actual_parameters.get(k) != EXPECTED_MODEL_PARAMETERS.get(k))
    raise RuntimeError(f'Baseline parameter compatibility mismatch: {differences}; versions={runtime_versions}')
print('FULL_MODEL_PARAMETERS_MATCH_S04')


## 5 全量训练

创建未拟合 Pipeline；只用训练 text 和监督标签。记录真实耗时和告警，不运行 CV/EDA。

In [ ]:
def fit_full_model(factory, train):
    model = factory()
    if hasattr(model.named_steps['classifier'], 'coef_'):
        raise ValueError('Factory must return an unfitted Pipeline')
    model.fit(train['text'].astype(str), train['model'])
    return model


def model_state_fingerprint(model):
    checksum = hashlib.sha256()
    for name, transform in model.named_steps['features'].transformer_list:
        checksum.update(name.encode('utf-8'))
        checksum.update(json.dumps(transform.vocabulary_, sort_keys=True, default=int).encode('utf-8'))
        checksum.update(transform.idf_.tobytes())
    for values in (model.named_steps['classifier'].coef_, model.named_steps['classifier'].intercept_):
        checksum.update(values.tobytes())
    return checksum.hexdigest()

with warnings.catch_warnings(record=True) as fit_warnings:
    warnings.simplefilter('always')
    started = time.perf_counter()
    model = fit_full_model(make_model, train)
    fit_seconds = time.perf_counter() - started
if model_config(model) != EXPECTED_MODEL_PARAMETERS or set(model.classes_) != set(class_order):
    raise RuntimeError('Fitted model parameters/classes differ from the verified baseline')
fitted_state = model_state_fingerprint(model)
for warning in fit_warnings:
    print('FIT_WARNING', warning.category.__name__, str(warning.message))
print('FULL_TRAIN_FINISHED', fit_seconds, 'seconds; warnings', len(fit_warnings))


## 6 运行时预测和提交契约

复用 S05 预测及严格提交函数。只以运行时 test.csv 的 id/text 为准，不调用 fit；决策分数按 class_order 对齐并留在内存，不作为概率或额外提交列。

In [ ]:
def validate_submission(submission, test, labels):
    if submission.columns.tolist() != ['id', 'model']:
        raise ValueError('Submission columns must be exactly id, model in that order')
    for column in ('id', 'model'):
        values = submission[column]
        if values.isna().any():
            raise ValueError(f'Submission {column} contains null values')
        if not pd.api.types.is_string_dtype(values.dtype) or not all(isinstance(v, str) for v in values):
            raise ValueError(f'Submission {column} must contain strings')
        if values.str.strip().eq('').any():
            raise ValueError(f'Submission {column} contains empty/whitespace strings')
    if submission['id'].duplicated().any():
        raise ValueError('Submission contains duplicate ID')
    missing = set(test['id']) - set(submission['id'])
    extra = set(submission['id']) - set(test['id'])
    if missing or extra or len(submission) != len(test):
        raise ValueError(f'Submission ID coverage mismatch: missing={sorted(missing)[:5]}, extra={sorted(extra)[:5]}')
    if submission['id'].tolist() != test['id'].tolist():
        raise ValueError('Submission ID order differs from test input')
    unknown = set(submission['model']) - set(labels)
    if unknown:
        raise ValueError(f'Submission contains illegal model labels: {sorted(unknown)}')
    return submission


def write_and_verify_submission(submission, test, labels, path, protected_roots=()):
    path = Path(path).expanduser().resolve()
    if path.name != 'submission.parquet':
        raise ValueError('Output filename must be submission.parquet; sample files are never overwritten')
    if any(path.is_relative_to(Path(root).resolve()) for root in protected_roots):
        raise ValueError(f'Output is inside a protected input/experiment directory: {path}')
    if path.exists():
        raise FileExistsError(f'Refusing to overwrite existing output: {path}')
    validate_submission(submission, test, labels)
    path.parent.mkdir(parents=True, exist_ok=True)
    # Exclusive creation prevents a concurrent writer from being overwritten.
    with path.open('xb') as stream:
        submission.to_parquet(stream, index=False)
    if path.stat().st_size == 0:
        raise ValueError(f'Empty submission file: {path}')
    loaded = pd.read_parquet(path)
    validate_submission(loaded, test, labels)
    pd.testing.assert_frame_equal(submission, loaded)
    return loaded
def predict_test(model, test, ns=None, batch_size=256):
    if ns is None:
        ns = {'validate_test': validate_test, 'class_order': class_order}
    ns['validate_test'](test, 'runtime test input')
    if batch_size <= 0:
        raise ValueError('batch_size must be positive')
    order, classes = list(ns['class_order']), list(model.classes_)
    if len(classes) != len(order) or set(classes) != set(order):
        raise ValueError('Model classes differ from official labels')
    columns = [classes.index(label) for label in order]
    predictions, decisions = [], []
    for start in range(0, len(test), batch_size):
        texts = test.iloc[start:start + batch_size]['text'].astype(str)
        predictions.extend(model.predict(texts).tolist())
        decisions.append(model.decision_function(texts)[:, columns])
    pred = np.asarray(predictions, dtype=object)
    scores = np.concatenate(decisions) if decisions else np.empty((0, len(order)), dtype=float)
    if scores.shape != (len(test), len(order)) or not np.isfinite(scores).all():
        raise ValueError('Invalid runtime decision score matrix')
    if len(pred) != len(test) or not set(pred).issubset(order):
        raise ValueError('Invalid prediction labels/count')
    return pred, scores


def build_submission(test, predictions, labels):
    if len(test) != len(predictions):
        raise ValueError('Prediction count differs from test rows')
    result = pd.DataFrame({'id': test['id'].tolist(), 'model': list(predictions)}, dtype='string')
    return validate_submission(result, test, labels)

with warnings.catch_warnings(record=True) as prediction_warnings:
    warnings.simplefilter('always')
    started = time.perf_counter()
    predictions, decision_scores = predict_test(model, test, {'validate_test': validate_test, 'class_order': class_order})
    prediction_seconds = time.perf_counter() - started
if model_state_fingerprint(model) != fitted_state:
    raise RuntimeError('Prediction changed the fitted model state')
submission = build_submission(test, predictions, class_order)
for warning in prediction_warnings:
    print('PREDICT_WARNING', warning.category.__name__, str(warning.message))


## 7 写出、回读与结束标记

仅输出 submission.parquet；真实回读后检查两列字符串、完整唯一 ID、顺序、合法标签及写前一致。已有输出拒绝覆盖，重新运行请用新会话或仅清理自己生成的候选输出。

In [ ]:
submission_path = OUTPUT_DIR / 'submission.parquet'
reloaded_submission = write_and_verify_submission(submission, test, class_order, submission_path, [DATA_DIR])
runtime_summary = {'status': 'local_runtime_validated', 'n_train': len(train), 'n_prompts': int(train['prompt_id'].nunique()),
    'n_test': len(test), 'fit_seconds': fit_seconds, 'prediction_seconds': prediction_seconds,
    'fit_warning_count': len(fit_warnings), 'prediction_warning_count': len(prediction_warnings),
    'python': platform.python_version(), 'versions': runtime_versions, 'class_order': list(class_order),
    'model_parameters_sha256': hashlib.sha256(json.dumps(actual_parameters, sort_keys=True).encode()).hexdigest(),
    'train_sha256': train_fingerprint['sha256'], 'test_sha256': test_fingerprint['sha256'],
    'submission_path': str(submission_path), 'submission_sha256': file_fingerprint(submission_path)['sha256'],
    'write_read_equal': True, 'fitted_state_unchanged': True}
print('SUBMISSION_VALIDATED', json.dumps(runtime_summary, ensure_ascii=False))
